# EyeVision AI — GPU training on Kaggle (optional)

The project already ships a disease model trained on ODIR-5K (CPU, 224 px). Use this notebook to
retrain **at higher resolution on a free Kaggle GPU** and get stronger models.

**Setup (one time, in the Kaggle notebook editor):**
1. *Add Input* → search **"Ocular Disease Recognition" (andrewmvd/ocular-disease-recognition-odir5k)** → Add.
2. *Add Input* → *Upload* → upload **EyeVisionAI.zip** as a private dataset named `eyevisionai-code`.
3. *Settings* → Accelerator **GPU T4 x2** (or P100) and **Internet on** (for ImageNet weights).
4. *Run All*. When it finishes, download `/kaggle/working/eyevision_outputs.zip` from the Output panel
   and unzip it over your project folder (it contains `models/` and `reports/`).

Research/educational use only — not a medical device.

In [ ]:
import glob, os, shutil, subprocess, zipfile
CODE_ZIP = (glob.glob('/kaggle/input/**/EyeVisionAI.zip', recursive=True) or [None])[0]
ODIR_ROOT = '/kaggle/input/ocular-disease-recognition-odir5k'
assert CODE_ZIP, 'Upload EyeVisionAI.zip as a Kaggle dataset first (see step 2).'
assert os.path.isdir(ODIR_ROOT), 'Add the ODIR-5K dataset as an input first (see step 1).'
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall('/kaggle/working')
os.chdir('/kaggle/working/EyeVisionAI')
!pip install -q timm
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) per-eye index from the diagnostic keywords + cached 384 px crops
!python scripts/prepare_odir.py --root {ODIR_ROOT} --size 384

In [ ]:
# 2) simple non-deep-learning baselines (same patient-level split)
!python baselines.py --task disease --dataset odir

In [ ]:
# 3) EfficientNet-B0 and ResNet-50, class-weighted BCE, 384 px
!python train.py --task disease --dataset odir --arch efficientnet_b0 --img-size 384 --epochs 20 --batch-size 32 --num-workers 2 --tag gpu384
!python train.py --task disease --dataset odir --arch resnet50 --img-size 384 --epochs 20 --batch-size 32 --num-workers 2 --tag gpu384

In [ ]:
# 4) optional: focal loss variant and EfficientNet-B2
!python train.py --task disease --dataset odir --arch efficientnet_b0 --img-size 384 --epochs 20 --loss focal --num-workers 2 --tag gpu384_focal
!python train.py --task disease --dataset odir --arch efficientnet_b2 --img-size 384 --epochs 20 --batch-size 24 --num-workers 2 --tag gpu384

In [ ]:
# 5) evaluate every new model on the held-out test patients, then compare
import glob
for ck in sorted(glob.glob('models/disease_*gpu384*.pth')):
    !python evaluate.py --task disease --checkpoint {ck}
!python evaluate.py --task disease --compare
!python scripts/loss_ablation.py --dataset odir --img-size 384

In [ ]:
# 6) package models + reports for download
out = '/kaggle/working/eyevision_outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for pat in ('models/*.pth', 'models/*.json', 'reports/**/*'):
        for f in glob.glob(pat, recursive=True):
            if os.path.isfile(f) and not f.endswith('predictions_test.csv'):
                z.write(f, f)
print('Download', out, f'({os.path.getsize(out)/1e6:.0f} MB)')